# Security Data — k-Nearest Neighbours (kNN)

## Dataset
Data sourced from `security_data_10000.csv` (synthetic data generated from original class dataset).

| Column | Description |
|--------|-------------|
| Sector | Industry sector of the company |
| CEO_Gender | Gender of the CEO (Male/Female) |
| Size | Company size (Large, Medium, Small) |
| Security_Invest | Security investment amount |
| Security_Breach_Att | Number of security breach attempts |
| Succ_Sec_Breaches | Number of successful security breaches |
| Sec_Rating | Overall security rating (High/Medium/Low) |
| CEO_Sec_Exp | CEO's level of security experience (High/Medium/Low) |
| LOT_in_Business | Length of time in business (years) |
| Stock_Market | Whether the company is listed on the stock market (Yes/No) |

---

## What is kNN?
kNN predicts based on the **k closest training examples** in feature space.
- **Low k** (e.g. k=1) — very sensitive to noise, high variance
- **High k** (e.g. k=15) — smoother decision boundary, lower variance but may underfit
- **Feature scaling is required** — kNN uses distance, so unscaled features will dominate

## This Notebook Covers
| Section | Task | Algorithm |
|---------|------|-----------|
| Part A | Predict Stock Market (Yes/No) | kNN Classifier (k=1,5,10,15) |
| Part B | Predict Successful Breaches | kNN Regressor (k=1,5,10,15) |

---

## Data Analysis Plan
1. **Load Data** — Read CSV
2. **Train-Test Split** — Split before feature engineering
3. **Create Label** — Binary (CEO Gender) or continuous (Succ_Sec_Breaches)
4. **Drop Target** — Remove target column from features
5. **Impute** — Handle missing values (fit on train only)
6. **Encode** — One-hot encode categorical features
7. **Scale** — StandardScaler (critical for kNN — distance-based algorithm)
8. **Fit & Predict** — Run kNN at k=1, 5, 10, 15
9. **Evaluate** — Confusion matrix + F1 (classifier) or R²/MSE (regressor)

# Part A — kNN Classifier
## Step 1 — Load Dataset

In [115]:
#LOAD THE DATASET


import pandas as pd

security_df = pd.read_csv("security_data_10000.csv")

security_df.head()

,Sector,CEO_Gender,Size,Security_Invest,Security_Breach_Att,Succ_Sec_Breaches,Sec_Rating,CEO_Sec_Exp,LOT_in_Business,Stock_Market
0,Hospitality,Male,Large,186,80,38,High,High,11,No
1,Hospitality,Female,Large,229,72,42,Medium,Medium,21,Yes
2,Hospitality,Male,Small,108,78,35,High,Medium,15,Yes
3,Hospitality,Male,Large,210,70,35,Medium,Low,14,No
4,Banking,Male,Small,34,19,3,Low,High,3,No


## Step 2 — Train-Test Split
Split performed **before** any feature engineering to prevent data leakage.

In [116]:
from sklearn.model_selection import train_test_split
security_df_train, security_df_test = train_test_split(security_df, test_size=0.2, random_state=42)


## Step 3 — Create Binary Label
`True` = Male CEO, `False` = Female CEO.

In [ ]:
security_df_train_label = (security_df_train["Stock_Market"] == 'Yes')

security_df_test_label = (security_df_test["Stock_Market"] =='Yes')

## Step 4 — Drop Target Column
`Stock_Market` removed from features — it is the target variable.

In [ ]:
security_df_train = security_df_train.drop(columns=["Stock_Market"])
security_df_test = security_df_test.drop(columns=["Stock_Market"])

In [119]:
security_df_train.head()


,Sector,Size,Security_Invest,Security_Breach_Att,Succ_Sec_Breaches,Sec_Rating,CEO_Sec_Exp,LOT_in_Business,Stock_Market
9254,Hospitality,Large,123,78,26,High,Medium,16,Yes
1561,Hospitality,Medium,419,250,90,Low,High,15,Yes
1670,Banking,Small,28,8,0,Medium,High,15,Yes
6087,Health Care,Large,86,49,26,High,Medium,18,No
6669,Health Care,Medium,125,16,5,Medium,High,10,Yes


In [120]:
import numpy as np  
import pandas as pd
from sklearn.impute import SimpleImputer


imputer = SimpleImputer(strategy="median")

security_df_train_num = security_df_train.select_dtypes(include=[np.number])
security_df_test_num = security_df_test.select_dtypes(include=[np.number])

imputer.fit(security_df_train_num)

security_df_train_tf = imputer.transform(security_df_train_num)
security_df_test_tf = imputer.transform(security_df_test_num)

security_df_train_tf = pd.DataFrame(security_df_train_tf, 
                                     columns=security_df_train_num.columns,
                                     index=security_df_train_num.index)

security_df_test_tf  = pd.DataFrame(security_df_test_tf, 
                                     columns=security_df_test_num.columns,
                                     index=security_df_test_num.index)



## Step 5 — Impute Numeric Features
Fit imputer on train only, transform both. Preserves column names and index for later concatenation.

In [ ]:
cat_cols = ['Sector', 'Size', 'Sec_Rating', 'CEO_Sec_Exp', 'Stock_Market']
numerical_cols = ["Security_Invest", "Security_Breach_Att", "Succ_Sec_Breaches", "LOT_in_Business"]

security_df_train_cat = pd.get_dummies(security_df_train[cat_cols], dtype=int)
security_df_test_cat  = pd.get_dummies(security_df_test[cat_cols],  dtype=int)

security_df_test_cat.head()

security_df_train_final = pd.concat([security_df_train_tf, security_df_train_cat], axis=1)
security_df_test_final  = pd.concat([security_df_test_tf,  security_df_test_cat],  axis=1)

print('Final train shape:', security_df_train_final.shape)
print('Final test shape: ', security_df_test_final.shape)

Final train shape: (8000, 18)
Final test shape:  (2000, 18)


## Step 6 — Encode Categorical Features
One-hot encode all categorical columns. `CEO_Gender` is excluded as it is the target.

In [122]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

security_df_train_scaled = scaler.fit_transform(security_df_train_final[numerical_cols])
security_df_test_scaled = scaler.transform(security_df_test_final[numerical_cols])

## After scaling, we need to convert the scaled arrays back to DataFrames and concatenate them with the categorical columns

security_df_train_final = pd.concat([security_df_train_final.drop(columns=numerical_cols), pd.DataFrame(security_df_train_scaled, columns=numerical_cols, index=security_df_train_final.index)], axis=1)
security_df_test_final = pd.concat([security_df_test_final.drop(columns=numerical_cols), pd.DataFrame(security_df_test_scaled, columns=numerical_cols, index=security_df_test_final.index)], axis=1)

security_df_train_final.head()


,Succ_Sec_Breaches,Sector_Banking,Sector_Health Care,Sector_Hospitality,Size_Large,Size_Medium,Size_Small,Sec_Rating_High,Sec_Rating_Low,Sec_Rating_Medium,CEO_Sec_Exp_High,CEO_Sec_Exp_Low,CEO_Sec_Exp_Medium,Stock_Market_No,Stock_Market_Yes,Security_Invest,Security_Breach_Att,LOT_in_Business
9254,26.0,0,0,1,1,0,0,1,0,0,0,0,1,0,1,0.508713,0.239649,0.414143
1561,90.0,0,0,1,0,1,0,0,1,0,1,0,0,0,1,4.093372,2.569869,0.275058
1670,0.0,1,0,0,0,0,1,0,0,1,1,0,0,0,1,-0.641769,-0.708697,0.275058
6087,26.0,0,1,0,1,0,0,1,0,0,0,0,1,1,0,0.060630,-0.153237,0.692313
6669,5.0,0,1,0,0,1,0,0,0,1,1,0,0,0,1,0.532933,-0.600315,-0.420367


## Step 7 — Scale Features
kNN uses Euclidean distance — scaling is **mandatory** so no single feature dominates. Fit scaler on train only.

In [123]:
from sklearn.neighbors import KNeighborsClassifier

knn_clf_1 = KNeighborsClassifier(n_neighbors=1)

knn_clf_5 = KNeighborsClassifier(n_neighbors=5)

knn_clf_10 = KNeighborsClassifier(n_neighbors=10)

knn_clf_15 = KNeighborsClassifier(n_neighbors=15)


knn_clf_1.fit(security_df_train_scaled, security_df_train_label)

knn_clf_5.fit(security_df_train_scaled, security_df_train_label)

knn_clf_10.fit(security_df_train_scaled, security_df_train_label)

knn_clf_15.fit(security_df_train_scaled, security_df_train_label)


security_df_test_pred_1 = knn_clf_1.predict(security_df_test_scaled)

security_df_test_pred_5 = knn_clf_5.predict(security_df_test_scaled)

security_df_test_pred_10 = knn_clf_10.predict(security_df_test_scaled)

security_df_test_pred_15 = knn_clf_15.predict(security_df_test_scaled)


from sklearn.metrics import confusion_matrix, classification_report

print("KNN with k=1")
print(confusion_matrix(security_df_test_label, security_df_test_pred_1))

print(classification_report(security_df_test_label, security_df_test_pred_1))


print("KNN with k=5")
print(confusion_matrix(security_df_test_label, security_df_test_pred_5))

print(classification_report(security_df_test_label, security_df_test_pred_5))

print("KNN with k=10")
print(confusion_matrix(security_df_test_label, security_df_test_pred_10))

print(classification_report(security_df_test_label, security_df_test_pred_10))

print("KNN with k=15")
print(confusion_matrix(security_df_test_label, security_df_test_pred_15))
print(classification_report(security_df_test_label, security_df_test_pred_15))




KNN with k=1
[[247 451]
 [471 831]]
              precision    recall  f1-score   support

       False       0.34      0.35      0.35       698
        True       0.65      0.64      0.64      1302

    accuracy                           0.54      2000
   macro avg       0.50      0.50      0.50      2000
weighted avg       0.54      0.54      0.54      2000

KNN with k=5
[[163 535]
 [310 992]]
              precision    recall  f1-score   support

       False       0.34      0.23      0.28       698
        True       0.65      0.76      0.70      1302

    accuracy                           0.58      2000
   macro avg       0.50      0.50      0.49      2000
weighted avg       0.54      0.58      0.55      2000

KNN with k=10
[[163 535]
 [326 976]]
              precision    recall  f1-score   support

       False       0.33      0.23      0.27       698
        True       0.65      0.75      0.69      1302

    accuracy                           0.57      2000
   macro avg       

k=15 is the best

## Step 8 — Fit kNN Classifier & Evaluate (k=1,5,10,15)
Higher k = smoother boundary. Evaluate with confusion matrix and classification report.

Accuracy increases slightly as k increases. However, recall for Female (False) drops at higher k — the model becomes biased toward predicting Male (the majority class). Overall F1 scores are low (~0.5), suggesting CEO Gender is not strongly predictable from these features.

# Part B — kNN Regressor
**Objective:** Predict `Succ_Sec_Breaches` (a continuous value) using all features.

kNN Regressor predicts by averaging the target values of the k nearest neighbours.

## Step 1 — Load Dataset

In [124]:
#LOAD THE DATASET


import pandas as pd

security_df = pd.read_csv("security_data_10000.csv")

security_df.head()

,Sector,CEO_Gender,Size,Security_Invest,Security_Breach_Att,Succ_Sec_Breaches,Sec_Rating,CEO_Sec_Exp,LOT_in_Business,Stock_Market
0,Hospitality,Male,Large,186,80,38,High,High,11,No
1,Hospitality,Female,Large,229,72,42,Medium,Medium,21,Yes
2,Hospitality,Male,Small,108,78,35,High,Medium,15,Yes
3,Hospitality,Male,Large,210,70,35,Medium,Low,14,No
4,Banking,Male,Small,34,19,3,Low,High,3,No


## Step 2 — Train-Test Split

In [125]:
from sklearn.model_selection import train_test_split
security_df_train, security_df_test = train_test_split(security_df, test_size=0.2, random_state=42)


## Step 3 — Create Continuous Label
`Succ_Sec_Breaches` is a numeric count — used directly as the regression target.

In [126]:
security_df_train_label = security_df_train["Succ_Sec_Breaches"]

security_df_test_label = security_df_test["Succ_Sec_Breaches"]

## Step 4 — Drop Target Column

In [127]:
security_df_train = security_df_train.drop(columns=["Succ_Sec_Breaches"])
security_df_test = security_df_test.drop(columns=["Succ_Sec_Breaches"])

## Step 5 — Impute, Encode & Scale

In [128]:
import numpy as np  
import pandas as pd
from sklearn.impute import SimpleImputer


imputer = SimpleImputer(strategy="median")

security_df_train_num = security_df_train.select_dtypes(include=[np.number])
security_df_test_num = security_df_test.select_dtypes(include=[np.number])

imputer.fit(security_df_train_num)

security_df_train_tf = imputer.transform(security_df_train_num)
security_df_test_tf = imputer.transform(security_df_test_num)

security_df_train_tf = pd.DataFrame(security_df_train_tf, 
                                     columns=security_df_train_num.columns,
                                     index=security_df_train_num.index)

security_df_test_tf  = pd.DataFrame(security_df_test_tf, 
                                     columns=security_df_test_num.columns,
                                     index=security_df_test_num.index)



In [129]:
cat_cols = ['Sector', 'CEO_Gender' ,'Size', 'Sec_Rating', 'CEO_Sec_Exp', 'Stock_Market']
numerical_cols = security_df_train_num.columns

print("Categorical columns:", cat_cols)
print("Numerical columns:", numerical_cols)


security_df_train_cat = pd.get_dummies(security_df_train[cat_cols], dtype=int)
security_df_test_cat  = pd.get_dummies(security_df_test[cat_cols],  dtype=int)

security_df_test_cat.head()

security_df_train_final = pd.concat([security_df_train_tf, security_df_train_cat], axis=1)
security_df_test_final  = pd.concat([security_df_test_tf,  security_df_test_cat],  axis=1)

print('Final train shape:', security_df_train_final.shape)
print('Final test shape: ', security_df_test_final.shape)

Categorical columns: ['Sector', 'CEO_Gender', 'Size', 'Sec_Rating', 'CEO_Sec_Exp', 'Stock_Market']
Numerical columns: Index(['Security_Invest', 'Security_Breach_Att', 'LOT_in_Business'], dtype='str')
Final train shape: (8000, 19)
Final test shape:  (2000, 19)


In [130]:
security_df_train_final.head()

,Security_Invest,Security_Breach_Att,LOT_in_Business,Sector_Banking,Sector_Health Care,Sector_Hospitality,CEO_Gender_Female,CEO_Gender_Male,Size_Large,Size_Medium,Size_Small,Sec_Rating_High,Sec_Rating_Low,Sec_Rating_Medium,CEO_Sec_Exp_High,CEO_Sec_Exp_Low,CEO_Sec_Exp_Medium,Stock_Market_No,Stock_Market_Yes
9254,123.0,78.0,16.0,0,0,1,0,1,1,0,0,1,0,0,0,0,1,0,1
1561,419.0,250.0,15.0,0,0,1,0,1,0,1,0,0,1,0,1,0,0,0,1
1670,28.0,8.0,15.0,1,0,0,0,1,0,0,1,0,0,1,1,0,0,0,1
6087,86.0,49.0,18.0,0,1,0,0,1,1,0,0,1,0,0,0,0,1,1,0
6669,125.0,16.0,10.0,0,1,0,0,1,0,1,0,0,0,1,1,0,0,0,1


In [131]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

security_df_train_scaled = scaler.fit_transform(security_df_train_final[numerical_cols])
security_df_test_scaled = scaler.transform(security_df_test_final[numerical_cols])

## After scaling, we need to convert the scaled arrays back to DataFrames and concatenate them with the categorical columns

security_df_train_final = pd.concat([security_df_train_final.drop(columns=numerical_cols), pd.DataFrame(security_df_train_scaled, columns=numerical_cols, index=security_df_train_final.index)], axis=1)
security_df_test_final = pd.concat([security_df_test_final.drop(columns=numerical_cols), pd.DataFrame(security_df_test_scaled, columns=numerical_cols, index=security_df_test_final.index)], axis=1)

security_df_train_final.head()


,Sector_Banking,Sector_Health Care,Sector_Hospitality,CEO_Gender_Female,CEO_Gender_Male,Size_Large,Size_Medium,Size_Small,Sec_Rating_High,Sec_Rating_Low,Sec_Rating_Medium,CEO_Sec_Exp_High,CEO_Sec_Exp_Low,CEO_Sec_Exp_Medium,Stock_Market_No,Stock_Market_Yes,Security_Invest,Security_Breach_Att,LOT_in_Business
9254,0,0,1,0,1,1,0,0,1,0,0,0,0,1,0,1,0.508713,0.239649,0.414143
1561,0,0,1,0,1,0,1,0,0,1,0,1,0,0,0,1,4.093372,2.569869,0.275058
1670,1,0,0,0,1,0,0,1,0,0,1,1,0,0,0,1,-0.641769,-0.708697,0.275058
6087,0,1,0,0,1,1,0,0,1,0,0,0,0,1,1,0,0.060630,-0.153237,0.692313
6669,0,1,0,0,1,0,1,0,0,0,1,1,0,0,0,1,0.532933,-0.600315,-0.420367


In [132]:
security_df_train_scaled

array([[ 5.08712744e-01,  2.39648596e-01,  4.14143129e-01],
       [ 4.09337160e+00,  2.56986891e+00,  2.75058077e-01],
       [-6.41768984e-01, -7.08696882e-01,  2.75058077e-01],
       ...,
       [ 1.22322245e+00, -1.93880766e-01, -3.11202805e-03],
       [-6.05437982e-01, -2.07428558e-01,  1.52682355e+00],
       [ 5.81374748e-01, -3.15810898e-01,  1.10956839e+00]],
      shape=(8000, 3))

## Step 6 — Fit kNN Regressor (k=1,5,10,15)
Evaluated using R² (closer to 1 = better) and MSE (closer to 0 = better).

In [133]:
from sklearn.neighbors import KNeighborsRegressor

knn_reg_1 = KNeighborsRegressor(n_neighbors=1)

knn_reg_5 = KNeighborsRegressor(n_neighbors=5)

knn_reg_10 = KNeighborsRegressor(n_neighbors=10)

knn_reg_15 = KNeighborsRegressor(n_neighbors=15)


knn_reg_1.fit(security_df_train_scaled, security_df_train_label)

knn_reg_5.fit(security_df_train_scaled, security_df_train_label)

knn_reg_10.fit(security_df_train_scaled, security_df_train_label)

knn_reg_15.fit(security_df_train_scaled, security_df_train_label)


security_df_test_pred_1 = knn_reg_1.predict(security_df_test_scaled)

security_df_test_pred_5 = knn_reg_5.predict(security_df_test_scaled)

security_df_test_pred_10 = knn_reg_10.predict(security_df_test_scaled)

security_df_test_pred_15 = knn_reg_15.predict(security_df_test_scaled)


In [134]:
from sklearn.metrics import r2_score, mean_squared_error


print("KNN with k=1")
print("R2 Score:", r2_score(security_df_test_label, security_df_test_pred_1))
print("MSE:", mean_squared_error(security_df_test_label, security_df_test_pred_1))

print("KNN with k=5")
print("R2 Score:", r2_score(security_df_test_label, security_df_test_pred_5))
print("MSE:", mean_squared_error(security_df_test_label, security_df_test_pred_5))

print("KNN with k=10")
print("R2 Score:", r2_score(security_df_test_label, security_df_test_pred_10))
print("MSE:", mean_squared_error(security_df_test_label, security_df_test_pred_10))

print("KNN with k=15")
print("R2 Score:", r2_score(security_df_test_label, security_df_test_pred_15))
print("MSE:", mean_squared_error(security_df_test_label, security_df_test_pred_15))



KNN with k=1
R2 Score: 0.8751046100721831
MSE: 167.4165
KNN with k=5
R2 Score: 0.9284741033089827
MSE: 95.87716
KNN with k=10
R2 Score: 0.933586192961963
MSE: 89.02464
KNN with k=15
R2 Score: 0.9355684470318127
MSE: 86.36751999999998


**Best result: k=10** (R²=0.928, MSE=155) — balances variance and bias well. k=1 overfits (memorises training data), while k=10 and k=15 start to underfit.